### $\alpha=\min\left(1,\frac{p}{q}\right)$

$$
\text{最终输出分布}=\text{大模型原本的输出分布}
$$
- 全变差距离：$D_{TV}(p, q)$

- https://chatgpt.com/c/69f2b780-c70c-83a7-855a-6710e8e64803
- https://aistudio.google.com/prompts/1lirgq9nnGn5PUB2Y_f-O8KKPG1s34eKm
- 如果小模型低估了某个词的概率（$q\lt p$） 
    - 小模型不太出这个词，但只要它出了，大模型觉得“这词很好”，会 $100\%$ 接受($\alpha=1$)。然而这还不够，大模型缺少的概率份额 $p-q$，会在大模型拒绝其他词后，通过重采样（Resampling）环节偷偷补齐。
- 如果小模型高估了某个词的概率（$q > p$）
    - 小模型太爱出这个词了。为了纠正它，大模型在验证时，会按比例 $\frac{p}{q}$砍掉多余的部分（拒绝掉）。剩下的概率刚好是 $p$。而且在重采样时，因为 $p-q\lt 0$，大模型绝不会再主动生成这个词。

### MTP

- 如何加速训练，早期为什么不用于推理
- 跟探测采样的区别是什么
- 为什么现在又开始用在推理加速了？

- 训练数据：$x_1,x_2,\ldots,x_T$ 
- 标准 next-token prediction 只优化：$p(x_{t+1}\mid x_{\le t})$
    - $L_{\text{next}}=-\frac{1}{T-1}\sum_{t=1}^{T-1}\log p_{\theta}(x_{t+1}\mid x_{\le t})$
    - 在位置 $t$，主模型只预测下一个 token $x_{t+1}$
- MTP 会额外优化：$p(x_{t+2}\mid x_{\le t}),\ p(x_{t+3}\mid x_{\le t}),\ldots$
- 通常写成：$L = L_{\text{next}} + \lambda \frac{1}{D}\sum_{k=1}^{D} L_{\text{future},k}$
    - MTP 额外加 $D$个“未来深度”的预测头或预测模块。若把第 $k$ 个 MTP 深度记为 $L_{\text{future},k}$，它预测的是更远的 token：
    - $D$ 额外预测的未来 token 深度数量。
    - $D=1$，主模型 $x_t \rightarrow x_{t+1}$，还额外预测一个更远 token：$(x_t,x_{t+1}) \rightarrow x_{t+2}$
    - $D=2$，主模型预测 $x_{t+1}$，两个 MTP 分别预测：$x_{t+2},\quad x_{t+3}$
    - $L=L_{\text{next}}+\frac{\lambda}{2}\left(L_{\text{future},1}+L_{\text{future},2}\right)$
- 含义
    - 每个位置不只给模型一个监督信号，而是给多个未来 token 的监督信号。
    - MTP 一方面能“densify training signals”，另一方面能迫使表示提前为未来 token 做规划。
    - 同样 token budget 下模型学得更充分，或者达到同等能力可能需要更少训练 token。
    - v3 默认配置 $D=1$，multi-token prediction depth 设置为 1，也就是除了精确的 next token，每个 token 再预测一个额外 token。
$$
D=1,\quad L_{\text{total}}=L_{\text{next}}+\lambda L_{\text{MTP}}^{1},\quad\lambda=\begin{cases}
0.3, & \text{前 }10T\text{ tokens}\\
0.1, & \text{后 }4.8T\text{ tokens}
\end{cases}
$$

- sequential MTP module，保留 causal chain。也就是第 $k$ 个深度不是从同一个 hidden state 并行乱猜远处 token，而是把前一深度表示和中间 token embedding 结合后继续往后推。论文称其区别于并行 independent output heads，并强调它 sequentially predicts additional tokens。

- 整个 MTP 机制在深度为 $k$ 的模块中，包含四个核心组件：共享嵌入层（Shared Embedding Layer）、共享输出头（Shared Output Head）、一个专属的 Transformer 块（$TRM_k$）以及一个线性投影矩阵 $M_k \in \mathbb{R}^{d \times 2d}$。其计算过程可严谨地分解为以下三个关键步骤：